But : instaurer un délai minimum entre deux locations. Un véhicule ne sera pas affiché dans les résultats de recherche si les heures d'arrivée ou de départ demandées sont trop proches d'une location déjà réservée
Cela résout le problème des départs tardifs, mais risque aussi de nuire aux revenus de Getaround et des propriétaires : nous devons trouver le juste compromis.

Seuil : quelle doit être la durée minimale du délai ?
Portée : faut-il activer cette fonctionnalité pour toutes les voitures ou uniquement pour les voitures connectées ?

Quelle part des revenus de notre propriétaire serait potentiellement affectée par cette fonctionnalité ?
Combien de locations seraient concernées par cette fonctionnalité, en fonction du seuil et de la portée que nous choisirons ?
À quelle fréquence les chauffeurs sont-ils en retard pour le prochain pointage ? Quel impact cela a-t-il sur le chauffeur suivant ?
Combien de cas problématiques cela résoudra-t-il en fonction du seuil et de la portée choisis ?

Tableau de bord Web

Fournir au moins un point de terminaison /predict . L'URL complète ressemblerait à ceci : https://your-url.com/predict.
Ce point de terminaison accepte les requêtes POST avec des données d'entrée JSON et doit renvoyer les prédictions. Nous supposons que les données d'entrée sont toujours correctement formatées . Par conséquent, la gestion des erreurs est inutile. Nous laissons cette gestion à votre discrétion.

Exemple de saisie :
{
  "input": [[7.0, 0.27, 0.36, 20.7, 0.045, 45.0, 170.0, 1.001, 3.0, 0.45, 8.8], [7.0, 0.27, 0.36, 20.7, 0.045, 45.0, 170.0, 1.001, 3.0, 0.45, 8.8]]
}
Copie
La réponse doit être un JSON avec une clé predictioncorrespondant à la prédiction.

Exemple de réponse :
{
  "prediction":[6,6]
}
Copie
page de documentation
Vous devez fournir aux utilisateurs une documentation sur votre API.

Il doit se trouver à l'adresse /docsde votre site web. Si nous reprenons l'exemple d'URL ci-dessus, il doit se trouver directement à https://your-url.com/docsl'adresse indiquée.

Cette brève documentation devrait au moins inclure :

Titre h1 : le titre est à votre discrétion.
Une description de chaque point de terminaison que l'utilisateur peut appeler, avec le nom du point de terminaison, la méthode HTTP, l'entrée requise et la sortie attendue (vous pouvez donner un exemple).
Vous pouvez ajouter toute autre information pertinente et styliser votre code HTML comme vous le souhaitez.

Production en ligne
Vous devez héberger votre API en ligne . Nous vous recommandons Hugging Face , car il est gratuit. Vous pouvez toutefois choisir n'importe quel autre fournisseur d'hébergement.

Une API en ligne documentée sur le serveur Hugging Face (ou tout autre fournisseur de votre choix) contenant au moins un /predictpoint de terminaison conforme à la description technique ci-dessus. Nous devrions pouvoir interroger ce point de terminaison de l'API /predicten utilisantcurl :
$ curl -i -H "Content-Type: application/json" -X POST -d '{"input": [[7.0, 0.27, 0.36, 20.7, 0.045, 45.0, 170.0, 1.001, 3.0, 0.45, 8.8]]}' http://your-url/predict
Copie
Ou Python :

import requests

response = requests.post("https://your-url/predict", json={
    "input": [[7.0, 0.27, 0.36, 20.7, 0.045, 45.0, 170.0, 1.001, 3.0, 0.45, 8.8]]
})
print(response.json())
Copie

In [9]:
import pandas as pd
import numpy as np
import plotly.io as pio

df = pd.read_excel("get_around_delay_analysis.xlsx")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21310 entries, 0 to 21309
Data columns (total 7 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   rental_id                                   21310 non-null  int64  
 1   car_id                                      21310 non-null  int64  
 2   checkin_type                                21310 non-null  object 
 3   state                                       21310 non-null  object 
 4   delay_at_checkout_in_minutes                16346 non-null  float64
 5   previous_ended_rental_id                    1841 non-null   float64
 6   time_delta_with_previous_rental_in_minutes  1841 non-null   float64
dtypes: float64(3), int64(2), object(2)
memory usage: 1.1+ MB


In [2]:
df.head()

,rental_id,car_id,checkin_type,state,delay_at_checkout_in_minutes,previous_ended_rental_id,time_delta_with_previous_rental_in_minutes
0,505000,363965,mobile,canceled,NaN,NaN,NaN
1,507750,269550,mobile,ended,-81.0,NaN,NaN
2,508131,359049,connect,ended,70.0,NaN,NaN
3,508865,299063,connect,canceled,NaN,NaN,NaN
4,511440,313932,mobile,ended,NaN,NaN,NaN


In [10]:
BG_COLOR = "#262626"
COLOR_RED = "#b91c1c"
COLOR_PINK = "#db2777"
pio.templates.default = "plotly_dark"

def style(fig):
    fig.update_layout(
        paper_bgcolor=BG_COLOR,
        plot_bgcolor=BG_COLOR,
        font_color="#ffffff",
        margin=dict(t=50, b=50, l=50, r=50)
    )
    fig.update_xaxes(showgrid=False)
    fig.update_yaxes(gridcolor="#333333")
    return fig

In [ ]:
df_clean = df.dropna(subset=['delay_at_checkout_in_minutes']).copy()

# 0 si en avance
df_clean['is_late'] = df_clean['delay_at_checkout_in_minutes'] > 0
df_clean['delay_amount'] = df_clean['delay_at_checkout_in_minutes'].clip(lower=0)

In [5]:
# Filtrer les locations qui se suivent
df_consecutive = df_clean[df_clean['previous_ended_rental_id'].notna()].copy()

# Calcul de la friction :
df_consecutive['real_friction'] = df_consecutive['delay_at_checkout_in_minutes'] > df_consecutive['time_delta_with_previous_rental_in_minutes']

friction_rate = df_consecutive['real_friction'].mean()
print(f"% de locations consécutives impactées par un retard : {friction_rate:.2%}")

% de locations consécutives impactées par un retard : 17.82%


In [6]:
# Comparaison mobile vs connect
comparison = df_clean.groupby('checkin_type')['delay_amount'].agg(['mean', 'median', 'max'])
print(comparison)

                    mean  median      max
checkin_type                             
connect        34.356261     0.0   1466.0
mobile        137.574706    14.0  71084.0


In [23]:
import plotly.express as px

# On filtre pour ne garder que les retards
df_graph = df_clean[df_clean['delay_amount'] > 0]

fig = px.histogram(
    df_graph, 
    x="delay_amount",
    color="checkin_type",
    barmode="overlay",
    nbins=500, 
    range_x=[0, 600],
    color_discrete_map={
        "mobile": COLOR_RED, 
        "connect": COLOR_PINK
    },
    labels={
        "checkin_type": "Type de Check-in",
        "delay_amount": "Retard au checkout (min)"
    },
    title="Analyse des retards"
)

fig.update_layout(
    yaxis_type="log",
    showlegend=True,
    legend=dict(
        title_text="",
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1,
        bgcolor="rgba(0,0,0,0)"
    )
)

fig.for_each_trace(lambda t: t.update(name=t.name.capitalize()))

fig = style(fig)
fig.show()

In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import joblib

df = pd.read_csv("get_around_pricing_project.csv")
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

X = df.drop(columns=['rental_price_per_day'])
y = df['rental_price_per_day']

# Preprocessing
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

# Pipeline
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=50, random_state=42))
])

# Training & sauvegarde
model.fit(X, y)
joblib.dump(model, "model.joblib")

['model.joblib']